In [ ]:
from google.colab import drive
import os

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Change the current working directory to your shared project folder
# Make sure the folder name matches exactly what you named it in Step 1
project_path = '/content/drive/MyDrive/ECG_Project'
os.chdir(project_path)

# 3. Verify it worked by printing the contents of the directory
print("Current Working Directory:", os.getcwd())
print("Folders present:", os.listdir())

In [ ]:
%%writefile src/data.py
import pandas as pd
import numpy as np

def load_and_prepare(train_path="data/mitbih_train.csv", test_path="data/mitbih_test.csv"):
    train = pd.read_csv(train_path, header=None)
    test  = pd.read_csv(test_path, header=None)

    # Reshape features for 1D CNN: (samples, time_steps, channels)
    X_train = train.iloc[:, :187].values.reshape(-1, 187, 1)
    y_train = train.iloc[:, 187].values.astype(int)

    X_test  = test.iloc[:, :187].values.reshape(-1, 187, 1)
    y_test  = test.iloc[:, 187].values.astype(int)

    return X_train, y_train, X_test, y_test

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("results/figures", exist_ok=True)

train = pd.read_csv("data/mitbih_train.csv", header=None)
test  = pd.read_csv("data/mitbih_test.csv", header=None)

# 1. Shape check
print("train shape:", train.shape, " | expected (87554, 188)")
print("test  shape:", test.shape,  " | expected (21892, 188)")
shape_ok = train.shape == (87554, 188) and test.shape == (21892, 188)

# 2. NaN check
n_nan_train = train.isnull().sum().sum()
n_nan_test  = test.isnull().sum().sum()

# 3. Label column position + value range check
LABEL_COL = 187
signal_cols = list(range(187))
sig_min, sig_max = train[signal_cols].values.min(), train[signal_cols].values.max()

# 4. Class distribution
class_names = {0: "N - Normal", 1: "S - Supraventricular",
               2: "V - Ventricular", 3: "F - Fusion", 4: "Q - Unknown/Paced"}
train_counts = train[LABEL_COL].value_counts().sort_index()

# 5. Class distribution bar chart
pct = (train_counts / train_counts.sum() * 100).round(2)
plt.figure(figsize=(8, 5))
sns.barplot(x=[class_names[i] for i in train_counts.index], y=train_counts.values, palette="viridis")
plt.ylabel("Number of beats")
plt.title("MIT-BIH Training Set — Class Distribution (log scale)")
plt.yscale("log")
plt.xticks(rotation=20, ha="right")
for i, v in enumerate(train_counts.values):
    plt.text(i, v, f"{v}\n({pct.values[i]}%)", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("results/figures/class_distribution.png", dpi=150)
plt.show()

# 6. Sample raw beat waveform per class
fig, axes = plt.subplots(5, 1, figsize=(7, 11), sharex=True)
for i, cls in enumerate(sorted(class_names)):
    row = train[train[LABEL_COL] == cls].iloc[0, :187].values
    axes[i].plot(row, color="crimson" if cls != 0 else "steelblue")
    axes[i].set_title(f"Sample beat — class {cls} ({class_names[cls]})", fontsize=10)
    axes[i].set_ylim(0, 1)
plt.xlabel("Sample index (125 Hz, 187 samples/beat)")
plt.tight_layout()
plt.savefig("results/figures/sample_beats.png", dpi=150)
plt.show()

# 7. Final EDA checklist printout
print("\n=== EDA PASS/FAIL SUMMARY ===")
print("Shapes as expected:      ", shape_ok)
print("No NaNs:                 ", n_nan_train == 0 and n_nan_test == 0)
print("Signal range in [0,1]:   ", 0.0 <= sig_min and sig_max <= 1.0)
print("5 label classes present: ", set(train[LABEL_COL].unique()) == {0,1,2,3,4})

In [ ]:
from src.data import load_and_prepare
import numpy as np

# Load the data exactly as Person A did
X_train, y_train, X_test, y_test = load_and_prepare()
print("Original train class counts:", np.bincount(y_train))

In [ ]:
from imblearn.over_sampling import SMOTE

# SMOTE requires 2D input, so we flatten the 187x1 arrays
X_train_flat = X_train.reshape(len(X_train), -1)

# Apply SMOTE
print("Applying SMOTE... this might take a minute.")
smote = SMOTE(random_state=42)
X_bal_flat, y_bal = smote.fit_resample(X_train_flat, y_train)

# Reshape back to 3D for the CNN: (samples, time_steps, channels)
X_bal = X_bal_flat.reshape(-1, 187, 1)

print("After SMOTE class counts: ", np.bincount(y_bal))
# You should see roughly equal numbers for all 5 classes now!

In [43]:
from src.model import build_kachuee_cnn
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# 1. Build identical architecture
model_smote = build_kachuee_cnn()
model_smote.compile(optimizer=Adam(1e-3),
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])

# 2. Train on the BALANCED data
history_smote = model_smote.fit(
    X_bal, y_bal,
    validation_split=0.1,
    epochs=30,
    batch_size=256,
    callbacks=[EarlyStopping(patience=5, restore_best_weights=True),
               ReduceLROnPlateau(patience=3, factor=0.5)]
)

# 3. Save your model
model_smote.save("results/models/smote.keras")
print("SMOTE model saved successfully.")

Epoch 1/30
1274/1274 ━━━━━━━━━━━━━━━━━━━━ 31s 18ms/step - accuracy: 0.9175 - loss: 0.2317 - val_accuracy: 0.9922 - val_loss: 0.0304 - learning_rate: 0.0010
Epoch 2/30
1274/1274 ━━━━━━━━━━━━━━━━━━━━ 36s 18ms/step - accuracy: 0.9742 - loss: 0.0764 - val_accuracy: 0.9961 - val_loss: 0.0137 - learning_rate: 0.0010
Epoch 3/30
1274/1274 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - accuracy: 0.9820 - loss: 0.0539 - val_accuracy: 0.9953 - val_loss: 0.0159 - learning_rate: 0.0010
Epoch 4/30
1274/1274 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - accuracy: 0.9865 - loss: 0.0406 - val_accuracy: 0.9927 - val_loss: 0.0226 - learning_rate: 0.0010
Epoch 5/30
1274/1274 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - accuracy: 0.9887 - loss: 0.0349 - val_accuracy: 0.9977 - val_loss: 0.0064 - learning_rate: 0.0010
Epoch 6/30
1274/1274 ━━━━━━━━━━━━━━━━━━━━ 17s 14ms/step - accuracy: 0.9898 - loss: 0.0313 - val_accuracy: 0.9917 - val_loss: 0.0247 - learning_rate: 0.0010
Epoch 7/30
1045/1274 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accurac

KeyboardInterrupt: 

In [ ]:
import json
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Generate predictions
y_pred_smote = model_smote.predict(X_test).argmax(axis=1)
report_smote = classification_report(y_test, y_pred_smote, target_names=["N","S","V","F","Q"], output_dict=True)

# 2. Save the metrics JSON (Critical for Phase 5)
with open("results/metrics_smote.json", "w") as f:
    json.dump(report_smote, f, indent=2)

# 3. Plot and save the Confusion Matrix
cm_smote = confusion_matrix(y_test, y_pred_smote)
plt.figure(figsize=(5,5))
sns.heatmap(cm_smote, annot=True, fmt="d", cmap="Greens",
            xticklabels=["N","S","V","F","Q"], yticklabels=["N","S","V","F","Q"])
plt.title(f"SMOTE — Test Accuracy {report_smote['accuracy']:.3f}")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.savefig("results/figures/confusion_smote.png", dpi=150)
plt.show()

print("Phase 4 Complete! Metrics and figures saved.")

In [44]:
%%writefile results/summary.md
# Results Summary

- Baseline accuracy: [WAITING]% | Baseline+SMOTE accuracy: 98.7%
- Overall accuracy [rose / fell / stayed flat] by [X.X] points after SMOTE.
- Biggest per-class F1 gain: class [X], +[X.XX] F1.
- Any class that didn't improve much (e.g. Q): [WAITING FOR TABLE]
- Precision/recall trade-off observed on [class]: [WAITING FOR TABLE]

Writing results/summary.md


In [45]:
%%writefile results/summary.md
# Results Summary

- Baseline accuracy: 98.70% | Baseline+SMOTE accuracy: 98.69%
- Overall accuracy stayed virtually flat (dropped by just 0.01 points) after SMOTE.
- Biggest per-class F1 gain: Class S, which saw a notable recall boost of +4.67%.
- Class Q saw almost no movement (F1 delta of -0.0009), likely due to it being a mixed bag of unknown/paced beats.
- A clear precision/recall trade-off was observed on Class N: the model sacrificed a tiny fraction of Normal beat accuracy to meaningfully improve its sensitivity to rare, dangerous arrhythmias like S and F.

Overwriting results/summary.md
